In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from openai import OpenAI
import os
api_key = os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [8]:
models = openai_client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-safeguard-20b
allam-2-7b
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
whisper-large-v3-turbo
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
openai/gpt-oss-120b


In [9]:
def llm(prompt):
    response = openai_client.responses.create(
        model="openai/gpt-oss-120b",
        input=prompt)
    return response.output_text

In [10]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely—you can enroll right away! Here’s how to get started:

1. **Visit the Udemy website** (or open the Udemy app).  
2. In the search bar, type **“Artificial Intelligence for Beginners: Learn AI in 10 Minutes.”**  
3. Click on the course title to open the course page.  
4. Press the **“Add to Cart”** button, then **“Proceed to Checkout.”**  
5. If you already have a Udemy account, sign in; if not, you’ll be prompted to create a free account (just an email and password).  
6. Complete the payment process, and the course will be added instantly to your “My Learning” library. You can start watching the videos right away on any device (desktop, tablet, or phone).

**Quick tip:** If you’d like a special discount, just let me know and I can share a promo code that will lower the price for you. Happy learning!


In [47]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [48]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [14]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. If you’d like to earn a certificate, just make sure to submit your project before the submission deadline. Otherwise, you can start learning and doing the homework right away.


**What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.**

In code, it looks like this:

``
 def rag(question):

    search_results = search(question)

    user_prompt = build_prompt(question, search_results)

    return llm(user_prompt)
    ``

That's the entire architecture. It comes down to three components.

The pieces are 
* search, 
* the prompt, 
and 
* the LLM:

search
prompt
LLM

**The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, so search quality matters a lot for RAG.**

In [15]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [49]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
response.raise_for_status()
courses_raw = response.json()


In [50]:
documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
len(documents)
documents[100]


{'id': '6698c6cbd4',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 1. Introduction to Machine Learning',
 'question': 'wget is not recognized as an internal or external command',
 'answer': 'If you encounter the error "wget is not recognized as an internal or external command," wget needs to be installed.\n\nThis error may also cause messages like "No such file or directory: \'output.csv.gz\'."\n\n**Installation Instructions:**\n\n- **On Ubuntu:**\n\n  ```bash\n  sudo apt-get install wget\n  ```\n\n- **On macOS:**\n\n  Use [Homebrew](https://brew.sh/):\n\n  ```bash\n  brew install wget\n  ```\n\n- **On Windows:**\n\n  Use [Chocolatey](https://chocolatey.org/):\n\n  ```bash\n  choco install wget\n  ```\n\n  Alternatively, download a binary from [GnuWin32](https://gnuwin32.sourceforge.net/packages/wget.htm) and place it in a location that is in your PATH (e.g., `C:/tools/`).\n\n**Alternative Windows Installation:**\n\n1. Download the latest wget binary for Windows from [ete

In [51]:
from minsearch import Index
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)


In [52]:
search_results = index.search("question",
             boost_dict={"question": 2, "section": 1, "answer": 1},
             filter_dict={"course": "llm-zoomcamp"},
             num_results=5)

In [61]:
def search(question,course="llm-zoomcamp"):
    boost_dict = {"question": 2, "section": 0.5}
    filter_dict = {"course": course}
    return index.search(question,
             boost_dict=boost_dict,           
             filter_dict=filter_dict,
             num_results=5)

In [26]:
search_results = search("I just discovered the course. Can I still join?")
search_results

[{'id': '5c4a8d2e60',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'Vector search: should I embed the question, the answer, or both?',
  'answer': 'There\'s no single right answer — it\'s an experiment to run on your dataset. The course shows three options:\n\n- Embed the answer (`text`) only — works because the model captures semantic similarity between questions and their answers.\n- Embed the question only — works because user queries look like the indexed questions.\n- Embed `question + " " + text` — often the best, but produces longer input and slightly more cost.\n\nPick whichever gives the best hit rate / MRR on your ground-truth set. The course materials include a side-by-side comparison.'},
 {'id': '341f71f28c',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': "How can I get structured output (Pydantic objects) from Gemini via the OpenAI-compatible endpoint when responses.parse isn't available?",
  'answer': 'To get p

In [27]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know." 
'''


In [28]:
USER_PROMPT_TEMPLATE = '''
question:
{question}

context:
{context}
'''

In [29]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: '+doc['question'])
        lines.append('A: '+doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [30]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

In [31]:
context = build_context(search_results)
context

'Module 2: Vector Search\nQ: Vector search: should I embed the question, the answer, or both?\nA: There\'s no single right answer — it\'s an experiment to run on your dataset. The course shows three options:\n\n- Embed the answer (`text`) only — works because the model captures semantic similarity between questions and their answers.\n- Embed the question only — works because user queries look like the indexed questions.\n- Embed `question + " " + text` — often the best, but produces longer input and slightly more cost.\n\nPick whichever gives the best hit rate / MRR on your ground-truth set. The course materials include a side-by-side comparison.\n\nModule 1: RAG\nQ: How can I get structured output (Pydantic objects) from Gemini via the OpenAI-compatible endpoint when responses.parse isn\'t available?\nA: To get parsed structured output, use the OpenAI SDK\'s chat-completions parsing flow instead of the newer Responses API. This is the right choice when you want to stay on the OpenAI 

In [32]:
USER_PROMPT_TEMPLATE.format(question = question, context = context)

'\nquestion:\nI just discovered the course. Can I join now?\n\ncontext:\nModule 2: Vector Search\nQ: Vector search: should I embed the question, the answer, or both?\nA: There\'s no single right answer — it\'s an experiment to run on your dataset. The course shows three options:\n\n- Embed the answer (`text`) only — works because the model captures semantic similarity between questions and their answers.\n- Embed the question only — works because user queries look like the indexed questions.\n- Embed `question + " " + text` — often the best, but produces longer input and slightly more cost.\n\nPick whichever gives the best hit rate / MRR on your ground-truth set. The course materials include a side-by-side comparison.\n\nModule 1: RAG\nQ: How can I get structured output (Pydantic objects) from Gemini via the OpenAI-compatible endpoint when responses.parse isn\'t available?\nA: To get parsed structured output, use the OpenAI SDK\'s chat-completions parsing flow instead of the newer Resp

In [36]:
question

'I just discovered the course. Can I join now?'

In [35]:
output = rag(question)
output

'**Short answer:**\u202fYes—you can join the DataTalks\u202fClub LLM Zoomcamp right now! The course is self‑paced and open‑enrollment, so you can start whenever you’re ready.\n\n---\n\n## How to get started\n\n| Step | What to do | Why it matters |\n|------|------------|----------------|\n| **1️⃣ Create a DataTalks\u202fClub account** | Go to the **[course landing page](https://datatalks.club/llm-zoomcamp)** and click **“Enroll / Join”**. Sign up with GitHub, Google, or email. | This gives you access to the repo, videos, notebooks, and the private Slack channel. |\n| **2️⃣ Clone the repository** | ```bash\\ngit clone https://github.com/DataTalksClub/llm-zoomcamp.git\\ncd llm-zoomcamp\\n``` | All lectures, assignments, and data live here. The repo is version‑controlled, so you’ll always have the exact material the instructors refer to. |\n| **3️⃣ Set up your environment** | • Install Python\u202f3.10+ <br>• Create a virtual environment (`uv venv`, `conda`, or `venv`) <br>• Install depen

In [34]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=prompt
)

In [35]:
response.output[0]

ResponseReasoningItem(id='resp_01m4k8tjbfezprbnn6gg4n1tvn', summary=[], type='reasoning', content=[Content(text='We need to answer: "I just discovered the course. Can I join now?" Based on context: "I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."\n\nThus answer: Yes, you can still join, but to get a certificate you need to submit your project before submissions close. Provide that.', type='reasoning_text')], encrypted_content=None, status='completed')

In [36]:
response.output[0].content[0].text

'We need to answer: "I just discovered the course. Can I join now?" Based on context: "I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."\n\nThus answer: Yes, you can still join, but to get a certificate you need to submit your project before submissions close. Provide that.'

In [37]:
response.output_text

'Yes—you can still join the course. Just keep in mind that if you’d like to earn a certificate, you’ll need to submit your project before the submission deadline (while we’re still accepting projects).'

In [38]:
response.usage

ResponseUsage(input_tokens=313, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0), output_tokens=135, output_tokens_details=OutputTokensDetails(reasoning_tokens=85), total_tokens=448)

In [39]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=prompt
)

In [41]:
def llm(instructions, user_prompt, model="openai/gpt-oss-120b"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

## FULL RAG

In [63]:
def rag(query, model="openai/gpt-oss-120b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [64]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes—you can still join the course. If you’d like to earn a certificate, just make sure you submit your capstone project while we’re still accepting submissions. Otherwise you can start learning right away.


In [65]:
rag("How do I get a certificate?")

'To earn a certificate you must:\n\n1. **Complete a live cohort** – certificates are only issued to students who finish the course while a live cohort is active. Self‑paced study alone isn’t enough.\n\n2. **Submit the Capstone project** – finish and submit your final project. You can work on the material and even prepare the project in self‑paced mode, but the actual submission must occur during a live cohort’s submission window.\n\n3. **Finish the required peer reviews** – after you submit the project, you’ll automatically receive peer‑review assignments. Complete all of them before the peer‑review deadline. No extra sign‑up is needed.\n\nHomework is optional; you can miss it and still receive the certificate as long as the capstone project and peer reviews are completed.\n\nOnce the peer‑review window closes, your score and the reviewers’ feedback will appear, and the certificate will be issued.'